# Stage 7 — FastAPI Endpoint for IDM-VTON LoRA Pipeline

Wraps the fine-tuned IDM-VTON pipeline as a REST API:
- `POST /tryon` — upload person_image + clothing_image (+ pre-computed agnostic/mask/pose) → returns JPEG
- `GET /health` — liveness check

**First version:** accepts pre-computed agnostic/mask/pose from Stage 1–4. Full end-to-end preprocessing is Stage 8.

**Runtime:** Google Colab (ngrok for external URL). Export `main.py` at the end for RunPod/Modal deployment.

**Confirmed working settings (Stage 6):**
- `guidance_scale=7.5`, `num_inference_steps=50`, `seed=789`
- `mask dilation_iters=12`, `feather_radius=15`
- LoRA path: `/content/drive/MyDrive/VirtualTryOn/checkpoints/stage6_lora/final`

## Cell 1 — Install dependencies

Same pin strategy as Stage 5/6: fix `huggingface_hub` first, then install the rest.

In [ ]:
# ── 1a. Pin huggingface_hub FIRST (Colab's system install is broken) ──────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'huggingface_hub==0.22.2', '--force-reinstall'], check=True)

# ── 1b. Core ML deps ─────────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.36.2',
    'diffusers==0.25.1',
    'accelerate==0.26.1',
    'peft==0.7.1',
    'xformers',
    'omegaconf',
    'einops',
    'torchvision',
], check=True)

# ── 1c. API + tunnel deps ─────────────────────────────────────────────────────
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'fastapi==0.110.0',
    'uvicorn[standard]==0.29.0',
    'python-multipart',   # needed for UploadFile
    'pyngrok==7.2.0',
    'nest-asyncio',
    'aiofiles',
    'httpx',              # for the test client
], check=True)

print('All deps installed.')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT  = '/content/drive/MyDrive/VirtualTryOn'
LORA_PATH   = f'{DRIVE_ROOT}/checkpoints/stage6_lora/final'
OUTPUT_DIR  = f'{DRIVE_ROOT}/output_images'

assert os.path.isdir(LORA_PATH), f'LoRA weights not found at {LORA_PATH}'
print(f'LoRA weights found: {os.listdir(LORA_PATH)}')

## Cell 3 — Clone IDM-VTON and patch deprecated API

In [ ]:
import os, subprocess

REPO = '/content/IDM-VTON'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', 'https://github.com/yisol/IDM-VTON', REPO], check=True)
else:
    print('Repo already cloned.')

# Patch: diffusers 0.25+ removed _remove_lora kwarg from Attention.set_processor
for fname in [
    f'{REPO}/src/unet_hacked_tryon.py',
    f'{REPO}/src/unet_hacked_garmnet.py',
]:
    src = open(fname).read()
    patched = src.replace(', _remove_lora=_remove_lora', '')
    if patched != src:
        open(fname, 'w').write(patched)
        print(f'Patched: _remove_lora removed from {os.path.basename(fname)}')
    else:
        print(f'No patch needed: {os.path.basename(fname)}')

import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('IDM-VTON repo ready.')

## Cell 4 — Load base IDM-VTON weights

Downloads ~15 GB on first run. Subsequent runs in the same session reuse them from `/content/`.

In [ ]:
import torch
from huggingface_hub import snapshot_download

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16
print(f'Device: {DEVICE}  |  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

BASE_MODEL_PATH = '/content/IDM-VTON-weights'
VAE_PATH        = '/content/sdxl-vae-fp16'

if not os.path.isdir(BASE_MODEL_PATH):
    print('Downloading IDM-VTON weights (~15 GB)...')
    snapshot_download('yisol/IDM-VTON', local_dir=BASE_MODEL_PATH)

if not os.path.isdir(VAE_PATH):
    print('Downloading SDXL VAE...')
    snapshot_download('madebyollin/sdxl-vae-fp16-fix', local_dir=VAE_PATH)

print('Weights ready.')

## Cell 5 — Build pipeline with LoRA weights

Loads VAE, both UNets, text encoders + tokenizers, scheduler, then merges LoRA adapter.

In [ ]:
import torch, os, types
from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer, CLIPVisionModelWithProjection, AutoProcessor
from diffusers import AutoencoderKL, DDPMScheduler, EulerDiscreteScheduler
from peft import PeftModel

from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
from src.unet_hacked_tryon   import UNet2DConditionModel as TryonUNet
from src.unet_hacked_garmnet import UNet2DConditionModel as GarmentUNet

# ── VAE ──────────────────────────────────────────────────────────────────────
vae = AutoencoderKL.from_pretrained(VAE_PATH, torch_dtype=DTYPE).to(DEVICE)

# ── Text encoders + tokenizers ───────────────────────────────────────────────
tokenizer   = CLIPTokenizer.from_pretrained(BASE_MODEL_PATH, subfolder='tokenizer')
tokenizer_2 = CLIPTokenizer.from_pretrained(BASE_MODEL_PATH, subfolder='tokenizer_2')
text_encoder   = CLIPTextModel.from_pretrained(BASE_MODEL_PATH, subfolder='text_encoder',   torch_dtype=DTYPE).to(DEVICE)
text_encoder_2 = CLIPTextModelWithProjection.from_pretrained(BASE_MODEL_PATH, subfolder='text_encoder_2', torch_dtype=DTYPE).to(DEVICE)

# ── CLIP vision (for garment image embeds) ───────────────────────────────────
image_encoder = CLIPVisionModelWithProjection.from_pretrained(BASE_MODEL_PATH, subfolder='image_encoder', torch_dtype=DTYPE).to(DEVICE)
image_processor = AutoProcessor.from_pretrained(BASE_MODEL_PATH, subfolder='image_encoder')

# ── Scheduler ─────────────────────────────────────────────────────────────────
noise_scheduler = EulerDiscreteScheduler.from_pretrained(BASE_MODEL_PATH, subfolder='scheduler')

# ── Garment UNet (frozen) ─────────────────────────────────────────────────────
unet_garment = GarmentUNet.from_pretrained(BASE_MODEL_PATH, subfolder='unet_encoder', torch_dtype=DTYPE).to(DEVICE)
unet_garment.eval()
for p in unet_garment.parameters(): p.requires_grad_(False)

# ── Tryon UNet — load base then merge LoRA ────────────────────────────────────
unet_tryon = TryonUNet.from_pretrained(BASE_MODEL_PATH, subfolder='unet', torch_dtype=DTYPE)
unet_tryon = PeftModel.from_pretrained(unet_tryon, LORA_PATH)
unet_tryon = unet_tryon.merge_and_unload()  # bake LoRA into weights, no peft overhead at inference
unet_tryon = unet_tryon.to(DEVICE)
unet_tryon.eval()

print('All model components loaded.')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Cell 6 — Assemble pipeline + pre-encode text prompts

Text encoders are released from VRAM after encoding to save memory (same trick as Stage 5/6).
Also applies the Resampler patch discovered in Stage 6.

In [ ]:
import torch

# ── Build pipeline ────────────────────────────────────────────────────────────
pipe = TryonPipeline(
    unet=unet_tryon,
    vae=vae,
    feature_extractor=image_processor,
    image_encoder=image_encoder,
    unet_encoder=unet_garment,
    scheduler=noise_scheduler,
    tokenizer=tokenizer,
    tokenizer_2=tokenizer_2,
    text_encoder=text_encoder,
    text_encoder_2=text_encoder_2,
)
pipe.enable_attention_slicing(1)
try:
    pipe.enable_xformers_memory_efficient_attention()
    print('xformers enabled')
except Exception as e:
    print(f'xformers unavailable ({e}), continuing without it')

# ── Pre-encode text prompts (done once, reused for every request) ────────────
PROMPT = 'a photo of a person wearing clothes, high quality, photorealistic'
NEG    = 'monochrome, lowres, bad anatomy, worst quality, low quality'

with torch.no_grad():
    (
        prompt_embeds, neg_embeds,
        pooled_embeds, neg_pooled
    ) = pipe.encode_prompt(
        PROMPT, num_images_per_prompt=1, do_classifier_free_guidance=True, negative_prompt=NEG
    )

# Release text encoders from VRAM
pipe.tokenizer = pipe.tokenizer_2 = None
pipe.text_encoder = pipe.text_encoder_2 = None
import gc; gc.collect(); torch.cuda.empty_cache()

# ── Resampler patch (Stage 6 finding) ─────────────────────────────────────────
# The pipeline passes raw CLIP hidden states (257×1280) to encoder_hid_proj (Resampler),
# but the projection forward() call is commented out in unet_hacked_tryon.py.
# Patch prepare_ip_adapter_image_embeds to run the projection so the UNet
# receives the correct projected embeds instead of raw CLIP states.
def _patched_prepare_ip_adapter_image_embeds(self, ip_adapter_image, device, num_images_per_prompt):
    with torch.no_grad():
        clip_out = self.image_encoder(
            self.feature_extractor(images=ip_adapter_image, return_tensors='pt').pixel_values.to(device, dtype=DTYPE)
        )
        hidden = clip_out.hidden_states[-2]  # (1, 257, 1280)
        if self.unet.encoder_hid_proj is not None:
            projected = self.unet.encoder_hid_proj(hidden)  # Resampler → (1, 16, 2048)
        else:
            projected = hidden
    if num_images_per_prompt > 1:
        projected = projected.repeat(num_images_per_prompt, 1, 1)
    return [projected]

pipe.prepare_ip_adapter_image_embeds = types.MethodType(_patched_prepare_ip_adapter_image_embeds, pipe)

print('Pipeline ready.')
print(f'VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Cell 7 — Mask utilities (expand + feather)

Confirmed working in Stage 6. Applied to every uploaded mask before inference.

In [ ]:
from PIL import Image, ImageFilter
import numpy as np

def expand_mask(mask_img: Image.Image, dilation_iters: int = 12, feather_radius: int = 15) -> Image.Image:
    """Expand + feather a binary mask. Excludes head region."""
    mask_arr = np.array(mask_img.convert('L'))
    rows = np.where(mask_arr.max(axis=1) > 128)[0]
    if len(rows) == 0:
        return mask_img
    top, bot = rows[0], rows[-1]
    h = bot - top
    head_cut = max(0, top - int(h * 0.05))
    bot_cut  = min(mask_img.size[1], bot + int(h * 0.10))

    expanded = mask_img.convert('L').point(lambda x: 255 if x > 128 else 0)
    for _ in range(dilation_iters):
        expanded = expanded.filter(ImageFilter.MaxFilter(7))

    arr = np.array(expanded)
    arr[:head_cut, :] = 0
    arr[bot_cut:, :]  = 0
    hard = Image.fromarray(arr)

    feat = hard.filter(ImageFilter.GaussianBlur(radius=feather_radius))
    feat_arr = np.array(feat).astype(np.float32)
    feat_arr[arr > 128] = 255.0
    return Image.fromarray(feat_arr.astype(np.uint8))


def expand_agnostic(agn_img: Image.Image, mask_exp: Image.Image) -> Image.Image:
    """Fill expanded mask region with neutral gray (128,128,128)."""
    arr = np.array(agn_img.convert('RGB')).copy()
    arr[np.array(mask_exp) > 128] = [128, 128, 128]
    return Image.fromarray(arr)


print('Mask utilities defined.')

## Cell 8 — Core inference function

All preprocessing + pipeline call in one function. Thread-safe via a `torch.no_grad()` context and a simple lock (set up in Cell 10).

In [ ]:
import torch
from PIL import Image

TARGET_W, TARGET_H = 768, 1024  # VITON-HD standard

def _resize_to_target(img: Image.Image, w: int = TARGET_W, h: int = TARGET_H) -> Image.Image:
    """Resize + center-crop to (w, h), preserving aspect ratio."""
    iw, ih = img.size
    scale = max(w / iw, h / ih)
    img = img.resize((int(iw * scale), int(ih * scale)), Image.LANCZOS)
    iw2, ih2 = img.size
    left = (iw2 - w) // 2
    top  = (ih2 - h) // 2
    return img.crop((left, top, left + w, top + h))


def run_tryon(
    person_img:    Image.Image,
    clothing_img:  Image.Image,
    agnostic_img:  Image.Image,
    mask_img:      Image.Image,
    pose_img:      Image.Image,
    guidance_scale: float = 7.5,
    num_steps:     int   = 50,
    seed:          int   = 789,
) -> Image.Image:
    """Run IDM-VTON inference. Returns RGB PIL image at TARGET_W×TARGET_H."""
    # ── Resize all inputs to target resolution ────────────────────────────────
    person_img   = _resize_to_target(person_img.convert('RGB'))
    clothing_img = _resize_to_target(clothing_img.convert('RGB'))
    agnostic_img = _resize_to_target(agnostic_img.convert('RGB'))
    mask_img     = _resize_to_target(mask_img.convert('L'))
    pose_img     = _resize_to_target(pose_img.convert('RGB'))

    # ── Expand mask + agnostic ────────────────────────────────────────────────
    mask_exp = expand_mask(mask_img)
    agnostic_exp = expand_agnostic(agnostic_img, mask_exp)

    # ── SDXL time ids ─────────────────────────────────────────────────────────
    add_time_ids = torch.tensor(
        [[TARGET_H, TARGET_W, 0, 0, TARGET_H, TARGET_W]],
        dtype=DTYPE, device=DEVICE
    )
    neg_time_ids = add_time_ids.clone()

    generator = torch.Generator(device=DEVICE).manual_seed(seed)

    with torch.no_grad():
        result = pipe(
            image=agnostic_exp,
            mask_image=mask_exp,
            pose_img=pose_img,
            cloth=clothing_img,
            ip_adapter_image=clothing_img,
            prompt_embeds=prompt_embeds,
            negative_prompt_embeds=neg_embeds,
            pooled_prompt_embeds=pooled_embeds,
            negative_pooled_prompt_embeds=neg_pooled,
            num_inference_steps=num_steps,
            guidance_scale=guidance_scale,
            height=TARGET_H,
            width=TARGET_W,
            generator=generator,
            add_time_ids=add_time_ids,
            negative_add_time_ids=neg_time_ids,
        ).images[0]

    return result


print('run_tryon() defined. Ready to wrap in FastAPI.')

## Cell 9 — Quick smoke-test of the inference function

Uses the test pair from Stage 5/6 to confirm the pipeline still works before starting the server.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

TEST_PERSON   = f'{DRIVE_ROOT}/input_images/test_person.jpg'
TEST_CLOTH    = f'{DRIVE_ROOT}/input_images/clothing.jpg'
TEST_AGNOSTIC = f'{DRIVE_ROOT}/output_images/agnostic_person.png'
TEST_MASK     = f'{DRIVE_ROOT}/output_images/erase_mask.png'
TEST_POSE     = f'{DRIVE_ROOT}/output_images/pose_output.jpg'

for p in [TEST_PERSON, TEST_CLOTH, TEST_AGNOSTIC, TEST_MASK, TEST_POSE]:
    assert os.path.exists(p), f'Missing test file: {p}'

result = run_tryon(
    person_img   = Image.open(TEST_PERSON),
    clothing_img = Image.open(TEST_CLOTH),
    agnostic_img = Image.open(TEST_AGNOSTIC),
    mask_img     = Image.open(TEST_MASK),
    pose_img     = Image.open(TEST_POSE),
)

result.save('/content/stage7_smoke_test.jpg', quality=92)
print('Smoke test passed.')

fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].imshow(Image.open(TEST_PERSON));  axes[0].set_title('Person');  axes[0].axis('off')
axes[1].imshow(Image.open(TEST_CLOTH));   axes[1].set_title('Clothing'); axes[1].axis('off')
axes[2].imshow(result);                   axes[2].set_title('Result');   axes[2].axis('off')
plt.tight_layout(); plt.show()

## Cell 10 — Write FastAPI app to disk

Written to `/content/app.py` — this same file is what you deploy to RunPod/Modal.

In [ ]:
APP_CODE = '''
"""
IDM-VTON LoRA — FastAPI inference server

POST /tryon
  Form fields:
    person_image   (file, required) — person photo
    clothing_image (file, required) — clothing product photo
    agnostic_image (file, required) — agnostic person image (shirt erased)
    mask_image     (file, required) — binary erase mask
    pose_image     (file, required) — pose keypoints overlay
    guidance_scale (float, default 7.5)
    num_steps      (int,   default 50)
    seed           (int,   default 789)

  Returns: JPEG image (Content-Type: image/jpeg)

GET /health
  Returns: {"status": "ok", "model": "IDM-VTON-LoRA"}
"""

import io, threading
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response
from PIL import Image

# These globals are injected by the notebook before the server starts.
# In standalone RunPod deployment, import them from model_loader.py instead.
import builtins
run_tryon       = getattr(builtins, '_stage7_run_tryon',       None)
expand_mask     = getattr(builtins, '_stage7_expand_mask',     None)
expand_agnostic = getattr(builtins, '_stage7_expand_agnostic', None)

if run_tryon is None:
    raise RuntimeError(
        "run_tryon not found in builtins. "
        "Run the notebook cells above before starting the server."
    )

# One request at a time — GPU is single-tenant
_gpu_lock = threading.Lock()

app = FastAPI(title="IDM-VTON LoRA API", version="0.1.0")


@app.get("/health")
def health():
    return {"status": "ok", "model": "IDM-VTON-LoRA"}


@app.post("/tryon")
async def tryon(
    person_image:   UploadFile = File(...),
    clothing_image: UploadFile = File(...),
    agnostic_image: UploadFile = File(...),
    mask_image:     UploadFile = File(...),
    pose_image:     UploadFile = File(...),
    guidance_scale: float = Form(7.5),
    num_steps:      int   = Form(50),
    seed:           int   = Form(789),
):
    for f, name in [
        (person_image, "person_image"),
        (clothing_image, "clothing_image"),
        (agnostic_image, "agnostic_image"),
        (mask_image, "mask_image"),
        (pose_image, "pose_image"),
    ]:
        if f.content_type not in ("image/jpeg", "image/png", "image/webp"):
            raise HTTPException(422, f"{name}: unsupported content type {f.content_type!r}")

    try:
        person_pil   = Image.open(io.BytesIO(await person_image.read()))
        clothing_pil = Image.open(io.BytesIO(await clothing_image.read()))
        agnostic_pil = Image.open(io.BytesIO(await agnostic_image.read()))
        mask_pil     = Image.open(io.BytesIO(await mask_image.read()))
        pose_pil     = Image.open(io.BytesIO(await pose_image.read()))
    except Exception as e:
        raise HTTPException(422, f"Could not decode image: {e}")

    if not _gpu_lock.acquire(blocking=True, timeout=300):
        raise HTTPException(503, "GPU busy — try again in a moment")
    try:
        result = run_tryon(
            person_img   = person_pil,
            clothing_img = clothing_pil,
            agnostic_img = agnostic_pil,
            mask_img     = mask_pil,
            pose_img     = pose_pil,
            guidance_scale = guidance_scale,
            num_steps      = num_steps,
            seed           = seed,
        )
    except Exception as e:
        raise HTTPException(500, f"Inference error: {e}")
    finally:
        _gpu_lock.release()

    buf = io.BytesIO()
    result.save(buf, format="JPEG", quality=92)
    return Response(content=buf.getvalue(), media_type="image/jpeg")
'''

with open('/content/app.py', 'w') as f:
    f.write(APP_CODE.strip())
print('Wrote /content/app.py')

## Cell 11 — Start the server + ngrok tunnel

Set your ngrok authtoken below. Get one free at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import builtins, nest_asyncio, uvicorn, threading
from pyngrok import ngrok, conf

# ── Inject inference function so app.py can reach it ─────────────────────────
builtins._stage7_run_tryon       = run_tryon
builtins._stage7_expand_mask     = expand_mask
builtins._stage7_expand_agnostic = expand_agnostic

# ── ngrok auth (replace with your token) ─────────────────────────────────────
NGROK_AUTHTOKEN = 'PASTE_YOUR_NGROK_AUTHTOKEN_HERE'
conf.get_default().auth_token = NGROK_AUTHTOKEN

PORT = 8000

# Open tunnel BEFORE starting uvicorn
public_url = ngrok.connect(PORT, 'http')
print(f'\nPublic URL: {public_url}')
print(f'  POST {public_url}/tryon')
print(f'  GET  {public_url}/health')

# ── Load app from file (keeps app.py self-contained) ─────────────────────────
import importlib.util, sys
spec = importlib.util.spec_from_file_location('app', '/content/app.py')
app_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app_mod)
fastapi_app = app_mod.app

# ── Run uvicorn in a background thread so the notebook stays interactive ─────
nest_asyncio.apply()
server_config = uvicorn.Config(fastapi_app, host='0.0.0.0', port=PORT, log_level='info')
server = uvicorn.Server(server_config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()
print('\nServer running. Proceed to Cell 12 to test it.')

## Cell 12 — Test the running endpoint

Sends the Stage 5/6 test pair to the local server and saves the response.

In [ ]:
import httpx, io, time
from PIL import Image
import matplotlib.pyplot as plt

BASE = f'http://localhost:{PORT}'

# ── Health check ──────────────────────────────────────────────────────────────
for _ in range(10):
    try:
        r = httpx.get(f'{BASE}/health', timeout=5)
        print('Health:', r.json()); break
    except Exception:
        time.sleep(1)
else:
    print('Server did not start in time.')

# ── Try-on request ────────────────────────────────────────────────────────────
with open(TEST_PERSON, 'rb') as p, \
     open(TEST_CLOTH,  'rb') as c, \
     open(TEST_AGNOSTIC, 'rb') as a, \
     open(TEST_MASK,   'rb') as m, \
     open(TEST_POSE,   'rb') as po:

    resp = httpx.post(
        f'{BASE}/tryon',
        files={
            'person_image':   ('person.jpg',   p,  'image/jpeg'),
            'clothing_image': ('cloth.jpg',    c,  'image/jpeg'),
            'agnostic_image': ('agnostic.png', a,  'image/png'),
            'mask_image':     ('mask.png',     m,  'image/png'),
            'pose_image':     ('pose.jpg',     po, 'image/jpeg'),
        },
        data={'guidance_scale': 7.5, 'num_steps': 50, 'seed': 789},
        timeout=300,
    )

print(f'Status: {resp.status_code}  |  Size: {len(resp.content)/1024:.1f} KB')
assert resp.status_code == 200, resp.text

result_api = Image.open(io.BytesIO(resp.content))
result_api.save('/content/stage7_api_test.jpg', quality=92)

fig, ax = plt.subplots(1, 1, figsize=(5, 7))
ax.imshow(result_api); ax.set_title('API result'); ax.axis('off')
plt.tight_layout(); plt.show()
print('Endpoint test passed.')

## Cell 13 — Save result + export `main.py` for RunPod

Copies the API result to Drive and writes a self-contained `main.py` that RunPod/Modal can run directly.

In [ ]:
import shutil

# Save API result to Drive
out_path = f'{OUTPUT_DIR}/stage7_api_result.jpg'
shutil.copy('/content/stage7_api_test.jpg', out_path)
print(f'Result saved to Drive: {out_path}')

# Export main.py for RunPod / Modal deployment
MAIN_PY = '''
"""
main.py — Standalone RunPod / Modal deployment entry point for IDM-VTON LoRA API

Required env vars:
  LORA_PATH   — path to LoRA weights directory  (default: /weights/lora)
  MODEL_PATH  — path to IDM-VTON base weights   (default: /weights/base)
  VAE_PATH    — path to SDXL VAE weights        (default: /weights/vae)
  PORT        — server port                      (default: 8000)

On RunPod: mount your Drive checkpoint as a volume at /weights.
"""
import os, io, sys, threading
import torch
import types
from PIL import Image, ImageFilter
import numpy as np
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response
from transformers import (CLIPTextModel, CLIPTextModelWithProjection,
                           CLIPTokenizer, CLIPVisionModelWithProjection, AutoProcessor)
from diffusers import AutoencoderKL, EulerDiscreteScheduler
from peft import PeftModel

REPO        = os.environ.get("REPO_PATH",  "/IDM-VTON")
MODEL_PATH  = os.environ.get("MODEL_PATH", "/weights/base")
VAE_PATH    = os.environ.get("VAE_PATH",   "/weights/vae")
LORA_PATH   = os.environ.get("LORA_PATH",  "/weights/lora")
PORT        = int(os.environ.get("PORT",   "8000"))
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE       = torch.float16
TARGET_W, TARGET_H = 768, 1024

sys.path.insert(0, REPO)
from src.tryon_pipeline import StableDiffusionXLInpaintPipeline as TryonPipeline
from src.unet_hacked_tryon   import UNet2DConditionModel as TryonUNet
from src.unet_hacked_garmnet import UNet2DConditionModel as GarmentUNet


def _load_models():
    vae = AutoencoderKL.from_pretrained(VAE_PATH, torch_dtype=DTYPE).to(DEVICE)
    tokenizer   = CLIPTokenizer.from_pretrained(MODEL_PATH, subfolder="tokenizer")
    tokenizer_2 = CLIPTokenizer.from_pretrained(MODEL_PATH, subfolder="tokenizer_2")
    text_encoder   = CLIPTextModel.from_pretrained(MODEL_PATH, subfolder="text_encoder",   torch_dtype=DTYPE).to(DEVICE)
    text_encoder_2 = CLIPTextModelWithProjection.from_pretrained(MODEL_PATH, subfolder="text_encoder_2", torch_dtype=DTYPE).to(DEVICE)
    image_encoder  = CLIPVisionModelWithProjection.from_pretrained(MODEL_PATH, subfolder="image_encoder", torch_dtype=DTYPE).to(DEVICE)
    image_processor = AutoProcessor.from_pretrained(MODEL_PATH, subfolder="image_encoder")
    scheduler = EulerDiscreteScheduler.from_pretrained(MODEL_PATH, subfolder="scheduler")
    unet_garment = GarmentUNet.from_pretrained(MODEL_PATH, subfolder="unet_encoder", torch_dtype=DTYPE).to(DEVICE).eval()
    for p in unet_garment.parameters(): p.requires_grad_(False)
    unet_tryon = TryonUNet.from_pretrained(MODEL_PATH, subfolder="unet", torch_dtype=DTYPE)
    unet_tryon = PeftModel.from_pretrained(unet_tryon, LORA_PATH).merge_and_unload().to(DEVICE).eval()
    pipe = TryonPipeline(
        unet=unet_tryon, vae=vae, feature_extractor=image_processor,
        image_encoder=image_encoder, unet_encoder=unet_garment, scheduler=scheduler,
        tokenizer=tokenizer, tokenizer_2=tokenizer_2,
        text_encoder=text_encoder, text_encoder_2=text_encoder_2,
    )
    pipe.enable_attention_slicing(1)
    try: pipe.enable_xformers_memory_efficient_attention()
    except Exception: pass

    PROMPT = "a photo of a person wearing clothes, high quality, photorealistic"
    NEG    = "monochrome, lowres, bad anatomy, worst quality, low quality"
    with torch.no_grad():
        pe, ne, pp, np_ = pipe.encode_prompt(PROMPT, num_images_per_prompt=1, do_classifier_free_guidance=True, negative_prompt=NEG)
    pipe.tokenizer = pipe.tokenizer_2 = pipe.text_encoder = pipe.text_encoder_2 = None

    def _patched_prepare(self, ip_adapter_image, device, num_images_per_prompt):
        with torch.no_grad():
            hidden = self.image_encoder(
                self.feature_extractor(images=ip_adapter_image, return_tensors="pt").pixel_values.to(device, dtype=DTYPE)
            ).hidden_states[-2]
            projected = self.unet.encoder_hid_proj(hidden) if self.unet.encoder_hid_proj else hidden
        if num_images_per_prompt > 1: projected = projected.repeat(num_images_per_prompt, 1, 1)
        return [projected]
    pipe.prepare_ip_adapter_image_embeds = types.MethodType(_patched_prepare, pipe)

    return pipe, (pe, ne, pp, np_)


def _expand_mask(mask_img, dilation_iters=12, feather_radius=15):
    arr = np.array(mask_img.convert("L"))
    rows = np.where(arr.max(axis=1) > 128)[0]
    if len(rows) == 0: return mask_img
    top, bot = rows[0], rows[-1]; h = bot - top
    hc = max(0, top - int(h*0.05)); bc = min(mask_img.size[1], bot + int(h*0.10))
    exp = mask_img.convert("L").point(lambda x: 255 if x > 128 else 0)
    for _ in range(dilation_iters): exp = exp.filter(ImageFilter.MaxFilter(7))
    a = np.array(exp); a[:hc,:]=0; a[bc:,:]=0; hard = Image.fromarray(a)
    feat_arr = np.array(hard.filter(ImageFilter.GaussianBlur(feather_radius))).astype(np.float32)
    feat_arr[a > 128] = 255.0
    return Image.fromarray(feat_arr.astype(np.uint8))


def _expand_agnostic(agn, mask_exp):
    a = np.array(agn.convert("RGB")).copy(); a[np.array(mask_exp) > 128] = [128,128,128]
    return Image.fromarray(a)


print("Loading models..."); pipe, (pe, ne, pp, np_) = _load_models(); print("Models ready.")
_lock = threading.Lock()


def _resize(img, w=TARGET_W, h=TARGET_H):
    iw, ih = img.size; s = max(w/iw, h/ih)
    img = img.resize((int(iw*s), int(ih*s)), Image.LANCZOS)
    iw2, ih2 = img.size
    return img.crop(((iw2-w)//2, (ih2-h)//2, (iw2-w)//2+w, (ih2-h)//2+h))


def _infer(person, clothing, agnostic, mask, pose, guidance_scale, num_steps, seed):
    person=_resize(person.convert("RGB")); clothing=_resize(clothing.convert("RGB"))
    agnostic=_resize(agnostic.convert("RGB")); mask=_resize(mask.convert("L")); pose=_resize(pose.convert("RGB"))
    mask_exp = _expand_mask(mask); agn_exp = _expand_agnostic(agnostic, mask_exp)
    tids = torch.tensor([[TARGET_H,TARGET_W,0,0,TARGET_H,TARGET_W]], dtype=DTYPE, device=DEVICE)
    gen  = torch.Generator(device=DEVICE).manual_seed(seed)
    with torch.no_grad():
        return pipe(image=agn_exp, mask_image=mask_exp, pose_img=pose, cloth=clothing,
                    ip_adapter_image=clothing, prompt_embeds=pe, negative_prompt_embeds=ne,
                    pooled_prompt_embeds=pp, negative_pooled_prompt_embeds=np_,
                    num_inference_steps=num_steps, guidance_scale=guidance_scale,
                    height=TARGET_H, width=TARGET_W, generator=gen,
                    add_time_ids=tids, negative_add_time_ids=tids.clone()).images[0]


app = FastAPI(title="IDM-VTON LoRA API", version="0.1.0")

@app.get("/health")
def health(): return {"status": "ok", "model": "IDM-VTON-LoRA"}

@app.post("/tryon")
async def tryon(
    person_image:   UploadFile = File(...),
    clothing_image: UploadFile = File(...),
    agnostic_image: UploadFile = File(...),
    mask_image:     UploadFile = File(...),
    pose_image:     UploadFile = File(...),
    guidance_scale: float = Form(7.5),
    num_steps:      int   = Form(50),
    seed:           int   = Form(789),
):
    try:
        p = Image.open(io.BytesIO(await person_image.read()))
        c = Image.open(io.BytesIO(await clothing_image.read()))
        a = Image.open(io.BytesIO(await agnostic_image.read()))
        m = Image.open(io.BytesIO(await mask_image.read()))
        po= Image.open(io.BytesIO(await pose_image.read()))
    except Exception as e: raise HTTPException(422, f"Image decode error: {e}")

    if not _lock.acquire(blocking=True, timeout=300): raise HTTPException(503, "GPU busy")
    try:
        result = _infer(p, c, a, m, po, guidance_scale, num_steps, seed)
    except Exception as e: raise HTTPException(500, f"Inference error: {e}")
    finally: _lock.release()

    buf = io.BytesIO(); result.save(buf, format="JPEG", quality=92)
    return Response(content=buf.getvalue(), media_type="image/jpeg")


if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")
'''

with open('/content/main.py', 'w') as f:
    f.write(MAIN_PY.strip())

# Save to Drive too
shutil.copy('/content/main.py', f'{DRIVE_ROOT}/main.py')
print('Saved main.py to Drive:', f'{DRIVE_ROOT}/main.py')
print('\nStage 7 complete!')

## Summary

| Cell | What it does |
|------|--------------|
| 1 | Install all deps (same pin strategy as Stage 5/6) |
| 2 | Mount Drive, verify LoRA weights path |
| 3 | Clone IDM-VTON, patch `_remove_lora` |
| 4 | Download base model weights (~15 GB, cached) |
| 5 | Load VAE, UNets, text encoders, merge LoRA |
| 6 | Build pipeline, pre-encode prompts, apply Resampler patch |
| 7 | Define `expand_mask` / `expand_agnostic` utilities |
| 8 | Define `run_tryon()` |
| 9 | Smoke test (same pair as Stage 5/6) |
| 10 | Write `/content/app.py` |
| 11 | Start uvicorn + ngrok tunnel |
| 12 | Test endpoint via `httpx` |
| 13 | Save result + export `main.py` for RunPod |

**Stage 8 next:** integrate Stage 1–4 preprocessing (pose, segmentation, agnostic generation) directly into the `/tryon` endpoint so raw photos work without pre-computed inputs.